# Training Data Overview

In [10]:
from pathlib import Path
import pandas as pd

data_path = Path("..") / "data" / f"training_pop30_genres.parquet"
df = pd.read_parquet(data_path)
print(f"Loaded {len(df):,} tracks from {data_path}")

Loaded 1,863,421 tracks from ../data/training_pop30_genres.parquet


## Basic Info

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1863421 entries, 0 to 1863420
Data columns (total 34 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   track_rowid             int64  
 1   track_name              object 
 2   artist_name             object 
 3   artist_rowid            int64  
 4   album_rowid             int64  
 5   album_name              object 
 6   album_type              object 
 7   label                   object 
 8   release_date            object 
 9   release_date_precision  object 
 10  id_isrc                 object 
 11  id_upc                  object 
 12  time_signature          int64  
 13  tempo                   float64
 14  key                     int64  
 15  mode                    int64  
 16  danceability            float64
 17  energy                  float64
 18  loudness                float64
 19  speechiness             float64
 20  acousticness            float64
 21  instrumentalness        float64

## Audio Features Statistics

In [50]:
audio_features = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence",
    "tempo", "time_signature", "key", "mode"
]

df[audio_features].describe()

,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,key,mode
count,1287.000000,1287.000000,1287.000000,1287.000000,1287.000000,1287.000000,1287.000000,1287.000000,1287.000000,1287.000000,1287.000000,1287.000000
mean,0.639603,0.650473,-6.719692,0.081230,0.225045,0.033389,0.172596,0.503746,120.918096,3.919192,5.369852,0.597514
std,0.155441,0.180578,3.943384,0.082529,0.246822,0.138178,0.132911,0.245567,26.502549,0.365286,3.580224,0.490590
min,0.093900,0.001740,-54.341000,0.023200,0.000008,0.000000,0.021900,0.000010,48.718000,1.000000,0.000000,0.000000
25%,0.532000,0.537000,-7.675500,0.034500,0.027900,0.000000,0.093900,0.307500,100.022500,4.000000,2.000000,0.000000
50%,0.657000,0.674000,-5.901000,0.047900,0.123000,0.000005,0.121000,0.497000,120.026000,4.000000,5.000000,1.000000
75%,0.755000,0.780500,-4.667500,0.090000,0.350000,0.000507,0.203500,0.708500,136.369000,4.000000,9.000000,1.000000
max,0.974000,0.988000,2.194000,0.600000,0.996000,0.995000,0.949000,0.981000,203.812000,5.000000,11.000000,1.000000


## Popularity & Metadata Statistics

In [51]:
metadata_cols = [
    "track_popularity", "album_popularity", "artist_popularity",
    "artist_followers", "duration_ms"
]

df[metadata_cols].describe()

,track_popularity,album_popularity,artist_popularity,artist_followers,duration_ms
count,1287.000000,1287.000000,1287.000000,1.287000e+03,1287.000000
mean,83.747475,77.236208,83.772339,2.343145e+07,212332.732712
std,2.848826,6.975752,8.351916,2.983998e+07,53639.122731
min,81.000000,51.000000,62.000000,3.356000e+03,60406.000000
25%,82.000000,72.000000,78.000000,4.055131e+06,177585.000000
50%,83.000000,76.000000,84.000000,1.131711e+07,208733.000000
75%,85.000000,82.000000,90.000000,2.961131e+07,240681.000000
max,100.000000,100.000000,100.000000,1.351401e+08,547733.000000


## Categorical Columns

In [52]:
print("Album types:")
print(df["album_type"].value_counts())
print()
print("Explicit:")
print(df["explicit"].value_counts())
print()
print("Mode (0=minor, 1=major):")
print(df["mode"].value_counts())

Album types:
album_type
album          969
single         313
compilation      5
Name: count, dtype: int64

Explicit:
explicit
0    862
1    425
Name: count, dtype: int64

Mode (0=minor, 1=major):
mode
1    769
0    518
Name: count, dtype: int64


## Genre Analysis

In [ ]:
# Tracks without genres
no_genre = df["genres"].isna() | (df["genres"] == "")
print(f"Tracks without genres: {no_genre.sum():,} ({no_genre.mean()*100:.1f}%)")

In [ ]:
# Most common genres
all_genres = df["genres"].dropna().str.split(";").explode()
print(f"Total genre tags: {len(all_genres):,}")
print(f"Unique genres: {all_genres.nunique():,}")
print()
print("Top 20 genres:")
all_genres.value_counts().head(20)

## Missing Values

In [53]:
missing = df.isna().sum()
missing[missing > 0]

genre_list    517
dtype: int64

## Top Artists by Track Count

In [44]:
df.groupby("artist_name").size().sort_values(ascending=False).head(20)

artist_name
Taylor Swift                  532
Drake                         361
The Weeknd                    263
Ariana Grande                 251
Pritam                        246
Eminem                        236
Stray Kids                    228
BTS                           222
$uicideboy$                   209
Future                        208
Anirudh Ravichander           198
The Beatles                   187
Kendrick Lamar                182
Lana Del Rey                  177
Henrique & Juliano            177
Beyoncé                       176
Kanye West                    175
Morgan Wallen                 166
SZA                           162
YoungBoy Never Broke Again    162
dtype: int64

## Release Date Distribution

In [45]:
# Extract year from release_date
df["release_year"] = pd.to_datetime(df["release_date"], errors="coerce").dt.year
df["release_year"].value_counts().sort_index().tail(20)

release_year
2006.0     1860
2007.0     1857
2008.0     1836
2009.0     1994
2010.0     2225
2011.0     2336
2012.0     2798
2013.0     3072
2014.0     3505
2015.0     4271
2016.0     5088
2017.0     6051
2018.0     7100
2019.0     8335
2020.0     9444
2021.0    11799
2022.0    13268
2023.0    18900
2024.0    29782
2025.0    21981
Name: count, dtype: int64

## Correlation Matrix (Audio Features)

In [46]:
df[audio_features].corr().round(2)

,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,key,mode
danceability,1.00,0.30,0.48,0.22,-0.30,-0.39,-0.09,0.53,-0.01,0.19,0.04,-0.11
energy,0.30,1.00,0.75,0.10,-0.71,-0.42,0.22,0.41,0.21,0.16,0.05,-0.11
loudness,0.48,0.75,1.00,0.09,-0.59,-0.69,0.10,0.41,0.20,0.18,0.05,-0.09
speechiness,0.22,0.10,0.09,1.00,-0.11,-0.14,0.06,0.09,0.04,0.06,0.01,-0.08
acousticness,-0.30,-0.71,-0.59,-0.11,1.00,0.38,-0.10,-0.24,-0.17,-0.16,-0.03,0.09
instrumentalness,-0.39,-0.42,-0.69,-0.14,0.38,1.00,-0.05,-0.36,-0.14,-0.13,-0.03,0.05
liveness,-0.09,0.22,0.10,0.06,-0.10,-0.05,1.00,0.02,0.02,0.01,-0.00,0.00
valence,0.53,0.41,0.41,0.09,-0.24,-0.36,0.02,1.00,0.11,0.09,0.05,-0.05
tempo,-0.01,0.21,0.20,0.04,-0.17,-0.14,0.02,0.11,1.00,0.00,0.01,0.00
time_signature,0.19,0.16,0.18,0.06,-0.16,-0.13,0.01,0.09,0.00,1.00,0.01,-0.04
